In [1]:
!pip install ultralytics > /dev/null 2>&1

from ultralytics import YOLO

# dataset_name = "bamboo11-1"

# data_yaml_content = f"""
# train: {'/kaggle/working/yolo_data/images/train'}
# val: {'/kaggle/working/yolo_data/images/val'}

# names:
#   0: "Bamboo"
#   1: "Joint"
# """

# with open('/kaggle/working/dataset.yaml', 'w') as f:
#     f.write(data_yaml_content)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# **创建dataset_yml**

In [2]:
import shutil
import os

dataset_name = "joint-11-3"
yolo_version = 'yolo11s'
project_root = '/kaggle/working/bamboo_only'
resume = False

data_yaml_content = f"""
train: {'/kaggle/working/bamboo_only/yolo_data/images/train'}
val:   {'/kaggle/working/bamboo_only/yolo_data/images/val'}

names:
  0: "Joint"
"""

# Ensure parent directory exists before writing dataset yaml
yaml_path = '/kaggle/working/bamboo_only/dataset.yaml'
yaml_dir = os.path.dirname(yaml_path)
if yaml_dir:
    try:
        os.makedirs(yaml_dir, exist_ok=True)
    except Exception:
        # If directory creation fails for any reason, continue and let open() raise if needed
        pass

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(data_yaml_content)


In [3]:

source_dir = f'/kaggle/input/{dataset_name}/dataset'
target_dir = '/kaggle/working/bamboo_only/dataset'

try:
    # 如果目标目录已存在，先删除
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
        print(f"已删除现有目录: {target_dir}")
    
    # 复制整个目录树
    shutil.copytree(source_dir, target_dir)
    print(f"成功复制 {source_dir} 到 {target_dir}")

    # Normalize image files (read and rewrite to ensure consistent format)
    try:
        normalize_images(target_dir)
    except Exception:
        pass

    # 验证复制结果
    if os.path.exists(target_dir):
        file_count = sum([len(files) for r, d, files in os.walk(target_dir)])
        print(f"目标目录包含 {file_count} 个文件")
        
except Exception as e:
    print(f"复制过程中出错: {e}")

成功复制 /kaggle/input/joint-11-3/dataset 到 /kaggle/working/bamboo_only/dataset
目标目录包含 3511 个文件


In [4]:
def download_with_wget():
    url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt"
    save_path = "/kaggle/working/yolo11s.pt"
    
    # 如果文件已存在，先删除
    if os.path.exists(save_path):
        os.remove(save_path)
        
    # 使用wget命令下载
    !wget -O {save_path} {url}
    
    # 检查是否下载成功
    if os.path.exists(save_path):
        file_size = os.path.getsize(save_path) / (1024 * 1024)
        return save_path
    else:
        return None

# 执行下载
model_path = download_with_wget()

--2025-11-03 14:08:47--  https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/1c11d5fd-cb03-4cf0-9eb2-0b0f5601ef45?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-11-03T14%3A55%3A53Z&rscd=attachment%3B+filename%3Dyolo11s.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-11-03T13%3A55%3A21Z&ske=2025-11-03T14%3A55%3A53Z&sks=b&skv=2018-11-09&sig=vn%2BovlaKa2h1rVzblL42RAHCYPIGRuRygXE%2FZCFZ8LA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2MjE4MDcyNywibmJmIjoxNzYyMTc4OTI3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvc

In [5]:
import os

model_path = '/kaggle/working/yolo11s.pt'
if not os.path.exists(model_path):
    raise SystemExit("stop")
else:
    print("OK")
    
model = YOLO("yolo11s.pt")

OK


In [6]:
import os
import shutil
import random
import yaml
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

# Clear directory if it exists
def clear_dir(directory):
    shutil.rmtree(directory, ignore_errors=True)
    print(f"Cleared directory: {directory}")
    os.makedirs(directory, exist_ok=True)

# Split dataset into training and testing sets
def train_test_split(path, dstPath, neg_path=None, split=0.2):
    """
    Robust train/val split that preserves original image filenames and supports
    multiple image extensions. Also skips non-file entries.
    """
    print("------ PROCESS STARTED -------")
    img_path = os.path.join(path, 'images')
    label_path = os.path.join(path, 'labels')

    # supported image extensions
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    # collect (base, filename) pairs for images
    files = []
    if not os.path.isdir(img_path):
        print(f"Image path does not exist: {img_path}")
        return

    for fname in os.listdir(img_path):
        fpath = os.path.join(img_path, fname)
        if not os.path.isfile(fpath):
            continue
        base, ext = os.path.splitext(fname)
        if ext.lower() in exts:
            files.append((base, fname))

    print(f"--- This folder has a total number of {len(files)} candidate images---")
    random.seed(42)
    random.shuffle(files)
    test_size = int(len(files) * split)
    train_size = len(files) - test_size

    train_path_img = os.path.join(dstPath, "images/train")
    train_path_label = os.path.join(dstPath, "labels/train")
    val_path_img = os.path.join(dstPath, "images/val")
    val_path_label = os.path.join(dstPath, "labels/val")

    os.makedirs(train_path_img, exist_ok=True)
    os.makedirs(train_path_label, exist_ok=True)
    os.makedirs(val_path_img, exist_ok=True)
    os.makedirs(val_path_label, exist_ok=True)

    copied_train = 0
    for base, fname in tqdm(files[:train_size]):
        if base == 'classes':
            continue
        src_img = os.path.join(img_path, fname)
        src_lbl = os.path.join(label_path, base + '.txt')
        if os.path.exists(src_lbl) and os.path.exists(src_img):
            shutil.copy2(src_img, os.path.join(train_path_img, fname))
            shutil.copy2(src_lbl, os.path.join(train_path_label, base + '.txt'))
            copied_train += 1
    print(f"------ Training data created with {copied_train} images -------")

    if neg_path:
        neg_images = [f for f in os.listdir(neg_path) if os.path.splitext(f)[1].lower() in exts]
        for fname in tqdm(neg_images):
            src_img = os.path.join(neg_path, fname)
            shutil.copy2(src_img, os.path.join(train_path_img, fname))
        print(f"------ Total {len(neg_images)} negative images added to the training data -------")

    copied_val = 0
    for base, fname in tqdm(files[train_size:]):
        if base == 'classes':
            continue
        src_img = os.path.join(img_path, fname)
        src_lbl = os.path.join(label_path, base + '.txt')
        if os.path.exists(src_lbl) and os.path.exists(src_img):
            shutil.copy2(src_img, os.path.join(val_path_img, fname))
            shutil.copy2(src_lbl, os.path.join(val_path_label, base + '.txt'))
            copied_val += 1
    print(f"------ Validation data created with {copied_val} images ----------")
    print("------ TASK COMPLETED -------")


def validate_images(image_dir, remove_bad=False, max_report=20):
    """
    Validate that images in image_dir are readable by OpenCV (cv2.imread).
    Returns a list of unreadable files. Optionally remove bad files.
    """
    bad_files = []
    if not os.path.isdir(image_dir):
        print(f"Validation skipped, directory does not exist: {image_dir}")
        return bad_files

    for root, _, files in os.walk(image_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            try:
                im = cv2.imread(fpath, cv2.IMREAD_UNCHANGED)
                if im is None:
                    bad_files.append(fpath)
                    if remove_bad:
                        os.remove(fpath)
            except Exception as e:
                bad_files.append(fpath)
                if remove_bad:
                    try:
                        os.remove(fpath)
                    except Exception:
                        pass

    if bad_files:
        print(f"Found {len(bad_files)} unreadable images under {image_dir}")
        for bf in bad_files[:max_report]:
            print(f" - {bf}")
        if len(bad_files) > max_report:
            print(f" ... and {len(bad_files) - max_report} more")
    else:
        print(f"All images under {image_dir} appear readable by OpenCV")

    return bad_files


def normalize_images(root_dir, exts=None, jpg_quality=95, png_compression=3):
    """
    Walk `root_dir`, read image files with OpenCV, normalize to 3-channel BGR uint8,
    drop alpha if present, ensure contiguous memory, and overwrite the original file.
    Silent on errors.
    """
    if exts is None:
        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    if not os.path.isdir(root_dir):
        return

    for root, _, files in os.walk(root_dir):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in exts:
                continue
            fpath = os.path.join(root, fname)
            try:
                # Try imread
                im = cv2.imread(fpath, cv2.IMREAD_UNCHANGED)
                # Try imdecode if imread fails
                if im is None:
                    try:
                        with open(fpath, 'rb') as f:
                            img_bytes = f.read()
                        img_array = np.frombuffer(img_bytes, np.uint8)
                        im = cv2.imdecode(img_array, cv2.IMREAD_UNCHANGED)
                    except Exception:
                        im = None
                # If still None, delete image and label
                if im is None:
                    try:
                        os.remove(fpath)
                        # Remove corresponding label file
                        base = os.path.splitext(fname)[0]
                        label_path = os.path.join(root, '../labels', base + '.txt')
                        if os.path.exists(label_path):
                            os.remove(label_path)
                    except Exception:
                        pass
                    continue
                # Convert 1-channel -> 3-channel
                if im.ndim == 2:
                    try:
                        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
                    except Exception:
                        continue
                # If has alpha channel (4), drop alpha
                if im.ndim == 3 and im.shape[2] == 4:
                    try:
                        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
                    except Exception:
                        # try another common ordering
                        try:
                            im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
                        except Exception:
                            continue
                # Ensure uint8
                if im.dtype != np.uint8:
                    im = np.clip(im, 0, 255).astype(np.uint8)
                im = np.ascontiguousarray(im)

                params = []
                if ext in ('.jpg', '.jpeg'):
                    params = [int(cv2.IMWRITE_JPEG_QUALITY), int(jpg_quality)]
                elif ext == '.png':
                    params = [int(cv2.IMWRITE_PNG_COMPRESSION), int(png_compression)]

                # Overwrite file
                try:
                    cv2.imwrite(fpath, im, params)
                except Exception:
                    # ignore write failures
                    pass
            except Exception:
                # silent on errors
                pass

def predict_with_trained_model(project_root, task, conf_threshold=0.4, imgsz=640):
    """
    使用训练好的模型进行预测
    
    参数:
        project_root: 项目根目录路径
        task: 任务类型 (detect, segment)
        conf_threshold: 置信度阈值，默认0.4
        imgsz: 图像尺寸，默认640
    """
    train_result_dir = os.path.join(project_root, "training_results")
    run_dir = os.path.join(project_root, "runs")
    
    # 加载超参数获取结果文件夹名称
    with open(os.path.join(project_root, "hyperparameters.yaml"), 'r', encoding='utf-8') as file:
        hyperparameters = yaml.safe_load(file)
    
    result_folder_name = hyperparameters['name']
    
    # 构建模型路径
    model_path = os.path.join(train_result_dir, result_folder_name, "weights/best.pt")
    
    if not os.path.exists(model_path):
        print(f"错误: 模型文件不存在: {model_path}")
        return False
    
    # 加载训练好的模型
    best_model = YOLO(model_path)
    
    # 进行预测
    test_images_dir = os.path.join(project_root, "test_images")
    if not os.path.exists(test_images_dir):
        print(f"警告: 测试图片目录不存在: {test_images_dir}")
        return False
    
    print(f"开始预测，使用模型: {model_path}")
    print(f"测试图片目录: {test_images_dir}")
    print(f"置信度阈值: {conf_threshold}")
    print(f"图像尺寸: {imgsz}")
    
    results = best_model.predict(source=test_images_dir,
                                imgsz=imgsz,
                                conf=conf_threshold,
                                save=True)
    
    # 复制预测结果到输出目录
    output_dir = os.path.join(project_root, 'output')
    shutil.copytree(os.path.join(run_dir, task), output_dir, dirs_exist_ok=True)
    
    print("------ 预测完成，结果已保存到输出目录 -------")
    return True


In [7]:
run_dir = os.path.join(project_root, "runs") # Directory to save training runs
train_result_dir = os.path.join(project_root, "training_results") # Directory to save training results
yolo_data_dir = os.path.join(project_root, "yolo_data") # YOLO formatted dataset path, store the split dataset here
dataset_dir = os.path.join(project_root, "dataset") # Original dataset path


# Check for existing checkpoint if resume is requested
# NOTE: train_result_dir already contains the 'training_results' segment, so avoid duplicating it.
last_model_path = os.path.join(train_result_dir, "weights/last.pt")

if resume and os.path.exists(last_model_path):
    print("=" * 60)
    print("🔄 RESUMING TRAINING FROM CHECKPOINT")
    print("=" * 60)
    print(f"Checkpoint path: {last_model_path}")
    
    # Load the existing model
    model = YOLO(last_model_path)
    
    # Resume training
    print("Resuming training with existing hyperparameters...")
    results = model.train(resume=True)
    
else:
    print("=" * 60)
    print("🚀 STARTING NEW TRAINING")
    print("=" * 60)
    
    # Create directories
    os.makedirs(yolo_data_dir, exist_ok=True)
    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(train_result_dir, exist_ok=True)

    # Clear directories for fresh start
    clear_dir(yolo_data_dir)
    clear_dir(run_dir)
    clear_dir(train_result_dir)

    # Split dataset into training and testing sets
    train_test_split(dataset_dir, dstPath=yolo_data_dir, split=0.2)

    # Validate images in the created yolo_data folders to catch unreadable/corrupt files
    validate_images(os.path.join(yolo_data_dir, 'images', 'train'))
    validate_images(os.path.join(yolo_data_dir, 'images', 'val'))
    
    # Initialize and train YOLO model
    model = YOLO(f"{yolo_version}.yaml")
    model.load(f"{yolo_version}.pt")

    # Train the model
    print("Starting new training with hyperparameters...")
    # Use an integer device (0) instead of a list [0]. Passing a list can break downstream
    # device handling inside ultralytics. Also disable automatic AMP checks (amp=False)
    # which invoke an internal fast-check that has been observed to pass non-numpy
    # objects into OpenCV's resize and raise the "src is not a numpy array" error.
    # If you want AMP, try enabling it after confirming the environment (cv2, Pillow)
    # and ultralytics versions are compatible.
    train_results = model.train(
        data='/kaggle/working/bamboo_only/dataset.yaml',
        epochs=600,
        imgsz=640,
        device=0,
        amp=False,
        augment=True,
        resume=False,  # 关键：添加resume参数
        degrees=10.0,
        translate=0.1,
        #scale=0.6,
        shear=0.05,
        perspective=0.0003,
        flipud=0.2,
        fliplr=0.2,
        mosaic=0.2,
        mixup=0.1,
        copy_paste=0.1,
        optimizer='AdamW',
        cos_lr=True,
        lr0=0.002,
        lrf=0.01,
        hsv_h=0.015,
        hsv_s=0.4,
        hsv_v=0.2,
        freeze=2,
        weight_decay=0.0001,
        patience=40
    )

🚀 STARTING NEW TRAINING
Cleared directory: /kaggle/working/bamboo_only/yolo_data
Cleared directory: /kaggle/working/bamboo_only/runs
Cleared directory: /kaggle/working/bamboo_only/training_results
------ PROCESS STARTED -------
--- This folder has a total number of 1755 candidate images---


100%|██████████| 1404/1404 [00:00<00:00, 2617.44it/s]


------ Training data created with 1404 images -------


100%|██████████| 351/351 [00:00<00:00, 2617.52it/s]


------ Validation data created with 351 images ----------
------ TASK COMPLETED -------
All images under /kaggle/working/bamboo_only/yolo_data/images/train appear readable by OpenCV
All images under /kaggle/working/bamboo_only/yolo_data/images/val appear readable by OpenCV
Transferred 499/499 items from pretrained weights
Starting new training with hyperparameters...
Ultralytics 8.3.223 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/bamboo_only/dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=600, erasing=0.4, exist_ok=False, fliplr=0.2, flipud=0.2, format=torchscript, fraction=1.0, freeze=2, half=Fal

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        351       1118      0.967      0.981      0.992      0.758
Speed: 0.1ms preprocess, 6.0ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /kaggle/working/runs/detect/train


## 压缩输出的检测结果，方便下载

In [8]:
import zipfile
import os

def zip_directory(directory_path, output_path):
    """
    压缩整个目录为ZIP文件
    """
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(directory_path):
            for file in files:
                file_path = os.path.join(root, file)
                # 在ZIP文件中创建相对路径
                arcname = os.path.relpath(file_path, directory_path)
                zipf.write(file_path, arcname)
    
    print(f"成功压缩 {directory_path} 为 {output_path}")

# 使用函数
source_dir = '/kaggle/working/runs/detect'
output_zip = '/kaggle/working/train.zip'

zip_directory(source_dir, output_zip)

成功压缩 /kaggle/working/runs/detect 为 /kaggle/working/train.zip
